# Panel Construction & Preliminary Analysis
## Port × HS4 × Country × Month — U.S. Imports (2013–2025)

This notebook:
1. Loads raw Census trade data (13 annual parquet files), NOAA Storm Events, and Atlantic HURDAT2 tracks.
2. Runs sanity checks on both datasets.
3. Builds a Port × HS4 × Country × Month panel with both broad state-level NOAA indicators and port-specific HURDAT2 wind-field exposure.
4. Analyses missing values and panel coverage.
5. Runs a preliminary event study around Hurricane Harvey (August 2017) using Houston and Galveston import volumes.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import numpy as np
import polars as pl

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in (candidate, *candidate.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find pyproject.toml")


PROJECT_ROOT = find_project_root()
CENSUS_DIR   = PROJECT_ROOT / "data" / "raw" / "census"
NOAA_DIR     = PROJECT_ROOT / "data" / "raw" / "noaa"
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
HURDAT_PATH  = INTERIM_DIR / "hurdat2_atlantic_tracks_2013_2025.parquet"
FIGURES_DIR  = PROJECT_ROOT / "outputs" / "figures"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

STUDY_YEARS = range(2013, 2026)   # 2013–2025 inclusive (13 years)

print(f"Project root : {PROJECT_ROOT}")
print(f"Census dir   : {CENSUS_DIR}")
print(f"NOAA dir     : {NOAA_DIR}")
print(f"Study period : {min(STUDY_YEARS)}–{max(STUDY_YEARS)}")


## 1. Load Raw Data

In [ ]:
# Load per-year Census parquet files and concatenate
census_frames = []
for year in STUDY_YEARS:
    matches = list(CENSUS_DIR.glob(f"census_imports_porths_hs4_country_expanded_53ports_{year}_*.parquet"))
    if matches:
        census_frames.append(pl.read_parquet(matches[0]))
    else:
        print(f"WARNING: No census file found for {year}")

imports_country_raw = pl.concat(census_frames, how="diagonal_relaxed")

# The country-detail files retain both individual partners and CTY_CODE='-' totals.
# Keep the total rows for the main Port × HS4 × Month panel; use CTY_CODE != '-'
# separately when constructing supplier-country concentration measures.
imports_raw = (
    imports_country_raw
    .filter((pl.col("COMM_LVL") == "HS4"))
    .sort(["time", "PORT", "I_COMMODITY"])
)

print(f"Census country-detail shape : {imports_country_raw.shape}")
print(f"Individual-country rows     : {imports_country_raw.filter(pl.col('CTY_CODE') != '-').height:,}")
print(f"All-country panel shape     : {imports_raw.shape}")
print(f"Time range           : {imports_raw['time'].min()} → {imports_raw['time'].max()}")
print(f"Unique ports         : {imports_raw['PORT'].n_unique()}")
print(f"Unique HS4 codes     : {imports_raw['I_COMMODITY'].n_unique()}")
print(f"Unique CTY_CODE      : {imports_raw['CTY_CODE'].n_unique()}")

imports_raw.head(3)


In [ ]:
noaa_file = next(NOAA_DIR.glob("noaa_storm_events_*.parquet"), None)
if noaa_file is None:
    raise FileNotFoundError(f"No NOAA storm events parquet found in {NOAA_DIR}")

storms_raw = (
    pl.read_parquet(noaa_file)
    .filter(pl.col("month").dt.year().is_between(2010, 2025))
    .with_columns(pl.col("month").dt.strftime("%Y-%m").alias("time"))
)

print(f"NOAA storms shape    : {storms_raw.shape}")
print(f"Time range           : {storms_raw['time'].min()} → {storms_raw['time'].max()}")
print(f"Event types          : {sorted(storms_raw['event_type'].unique().to_list())}")
print(f"Unique states        : {storms_raw['state'].n_unique()}")
storms_raw.head(3)


## 2. Sanity Checks

In [ ]:
# ── Census sanity checks ──────────────────────────────────────────────────────

print("=== Row counts by year ===")
print(
    imports_raw
    .with_columns(pl.col("time").str.slice(0, 4).cast(pl.Int32).alias("year"))
    .group_by("year").agg(pl.len().alias("rows"))
    .sort("year")
)

print("\n=== Missing values ===")
print(imports_raw.null_count())

print("\n=== GEN_VAL_MO summary (USD) ===")
print(imports_raw.select("GEN_VAL_MO").describe())

n_zero = imports_raw.filter(pl.col("GEN_VAL_MO").is_null() | (pl.col("GEN_VAL_MO") <= 0)).height
print(f"\nRows with GEN_VAL_MO null or ≤ 0 : {n_zero:,}  ({n_zero / imports_raw.height * 100:.2f}%)")

print("\n=== Unique ports per year ===")
print(
    imports_raw
    .with_columns(pl.col("time").str.slice(0, 4).alias("year"))
    .group_by("year").agg(pl.col("PORT").n_unique().alias("n_ports"))
    .sort("year")
)


In [ ]:
# ── NOAA sanity checks ────────────────────────────────────────────────────────

print("=== Event counts by year and type ===")
event_pivot = (
    storms_raw
    .group_by(["year", "event_type"])
    .agg(pl.len().alias("n"))
    .sort(["year", "event_type"])
    .pivot("event_type", index="year", values="n", aggregate_function="sum")
    .sort("year")
)
print(event_pivot)

print("\n=== Missing values in NOAA data ===")
print(storms_raw.null_count())

damage_events = storms_raw.filter(
    pl.col("property_damage_usd").is_not_null() & (pl.col("property_damage_usd") > 0)
)
print(f"\nEvents with property_damage_usd > 0 : {damage_events.height:,}")
print(damage_events.select("property_damage_usd").describe())

print("\n=== Top 10 states by event count ===")
print(
    storms_raw.group_by("state")
    .agg(pl.len().alias("n_events"))
    .sort("n_events", descending=True)
    .head(10)
)


## 3. Build Port × HS4 × Month Panel

This section keeps two distinct storm measures. `storm_month` is the original broad NOAA state-level screening indicator. `hurdat_storm_month` is the port-specific Atlantic HURDAT2 treatment: a port is exposed when its representative harbor point falls inside the storm's reported 34-kt wind quadrant. If all four 34-kt radii are missing for an observation, a 100-nautical-mile track buffer is used and separately flagged as a fallback. `hurdat_near_track_100nm_month` is retained only as a sensitivity measure. West Coast and Hawaii ports are marked `hurdat_atlantic_covered=False`; their zeros must not be interpreted as no tropical-cyclone exposure because this file contains only the Atlantic basin.

HURDAT2 and NOAA Storm Events do not have a reliable shared event identifier here, so they are not force-matched by ID. Both are joined independently to the panel through port and time.

In [ ]:
# Coastal state → Census Schedule D port-code mapping.
# This matches the expanded 53-port universe in census_download_colab.ipynb.
# Reference: https://www.census.gov/foreign-trade/schedules/d/distcode.html
STATE_PORT_MAP: dict[str, list[str]] = {
    "MASSACHUSETTS":  ["0401"],
    "RHODE ISLAND":   ["0502"],
    "CALIFORNIA":     ["2501", "2704", "2709", "2713", "2811", "2812"],
    "OREGON":         ["2904"],
    "WASHINGTON":     ["2905", "2908", "3001", "3002"],
    "NEW YORK":       ["1001"],
    "NEW JERSEY":     ["1003"],
    "PENNSYLVANIA":   ["1101"],
    "DELAWARE":       ["1103"],
    "MARYLAND":       ["1303"],
    "VIRGINIA":       ["1401"],
    "NORTH CAROLINA": ["1501", "1511"],
    "SOUTH CAROLINA": ["1601"],
    "GEORGIA":        ["1701", "1703"],
    "FLORIDA":        [
        "1801", "1803", "1805", "1816", "1818",
        "1819", "1821", "5201", "5203", "5204",
    ],
    "ALABAMA":        ["1901"],
    "MISSISSIPPI":    ["1902", "1903"],
    "LOUISIANA":      ["2002", "2004", "2010", "2017"],
    "TEXAS":          [
        "2101", "2102", "2103", "2104", "5301",
        "5306", "5310", "5311", "5312", "5313",
    ],
    "HAWAII":         ["3201"],
    "PUERTO RICO":    ["4909"],
}

MAPPED_PORTS = [port for ports in STATE_PORT_MAP.values() for port in ports]
assert len(MAPPED_PORTS) == 53, f'Expected 53 mapped ports, found {len(MAPPED_PORTS)}'
assert len(MAPPED_PORTS) == len(set(MAPPED_PORTS)), 'A port is mapped to multiple states'
assert all(len(port) == 4 and port.isdigit() for port in MAPPED_PORTS)
print(f'Mapped {len(MAPPED_PORTS)} Census ports across {len(STATE_PORT_MAP)} coastal states/territories')

state_port_df = pl.DataFrame(
    [{"state": state, "PORT": port}
     for state, ports in STATE_PORT_MAP.items()
     for port in ports]
)

# Aggregate storm events by PORT × month
storms_by_port = (
    storms_raw
    .with_columns(pl.col("state").str.to_uppercase())
    .join(state_port_df, on="state", how="inner")
    .group_by(["PORT", "time"])
    .agg(
        pl.len().alias("n_storm_events"),
        pl.col("property_damage_usd").sum().alias("total_property_damage_usd"),
        pl.col("crop_damage_usd").sum().alias("total_crop_damage_usd"),
    )
    .with_columns(pl.lit(1).cast(pl.Int8).alias("storm_month"))
    .sort(["PORT", "time"])
)

print(f"PORT × month combinations with storms : {storms_by_port.height:,}")
storms_by_port.head(5)


### 3.1 Port-specific HURDAT2 exposure

The representative points below locate the harbor or principal marine approach for each Census district. They are suitable for regional tropical-cyclone exposure, not berth-level analysis. The primary flag uses the reported 34-kt quadrant intersecting the port point. A 100-nm center-track rule is used only when all four 34-kt radii are missing; the unrestricted 100-nm result is also saved as a sensitivity flag. Because HURDAT2 positions are usually six-hourly, `n_exposed_track_points` is an observation count, not a claim about exposure hours.

In [ ]:
# Representative harbor / marine-approach points: PORT -> (longitude, latitude).
# Keep these points versioned: changing a point changes the treatment assignment.
PORT_REFERENCE_POINTS: dict[str, tuple[float, float]] = {
    "0401": (-71.03, 42.35),   # Boston
    "0502": (-71.40, 41.78),   # Providence
    "1001": (-74.03, 40.60),   # New York
    "1003": (-74.15, 40.68),   # Newark
    "1101": (-75.14, 39.91),   # Philadelphia
    "1103": (-75.52, 39.72),   # Wilmington, DE
    "1303": (-76.50, 39.20),   # Baltimore
    "1401": (-76.20, 36.98),   # Norfolk-Newport News
    "1501": (-77.95, 34.20),   # Wilmington, NC
    "1511": (-76.70, 34.72),   # Beaufort-Morehead City
    "1601": (-79.85, 32.76),   # Charleston
    "1701": (-81.49, 31.13),   # Brunswick
    "1703": (-80.90, 32.05),   # Savannah approach
    "1801": (-82.65, 27.65),   # Tampa Bay
    "1803": (-81.42, 30.38),   # Jacksonville
    "1805": (-81.47, 30.67),   # Fernandina
    "1816": (-80.60, 28.40),   # Port Canaveral
    "1818": (-85.67, 30.12),   # Panama City
    "1819": (-87.22, 30.40),   # Pensacola
    "1821": (-82.55, 27.64),   # Port Manatee
    "1901": (-88.05, 30.60),   # Mobile
    "1902": (-89.08, 30.36),   # Gulfport
    "1903": (-88.55, 30.35),   # Pascagoula
    "2002": (-90.05, 29.90),   # New Orleans
    "2004": (-91.18, 30.45),   # Baton Rouge
    "2010": (-90.70, 30.05),   # Gramercy
    "2017": (-93.25, 30.22),   # Lake Charles
    "2101": (-93.93, 29.88),   # Port Arthur
    "2102": (-93.84, 29.73),   # Sabine
    "2103": (-93.73, 30.09),   # Orange
    "2104": (-94.09, 30.08),   # Beaumont
    "2501": (-117.17, 32.68),  # San Diego
    "2704": (-118.25, 33.72),  # Los Angeles
    "2709": (-118.15, 33.73),  # Long Beach
    "2713": (-119.21, 34.14),  # Port Hueneme
    "2811": (-122.33, 37.80),  # Oakland
    "2812": (-122.37, 37.91),  # Richmond, CA
    "2904": (-122.76, 45.61),  # Portland
    "2905": (-122.95, 46.11),  # Longview
    "2908": (-122.70, 45.63),  # Vancouver, WA
    "3001": (-122.38, 47.60),  # Seattle
    "3002": (-122.47, 47.27),  # Tacoma
    "3201": (-157.87, 21.30),  # Honolulu
    "4909": (-66.12, 18.45),   # San Juan
    "5201": (-80.14, 25.77),   # Miami
    "5203": (-80.10, 26.09),   # Port Everglades
    "5204": (-80.05, 26.77),   # West Palm Beach
    "5301": (-95.20, 29.73),   # Houston Ship Channel
    "5306": (-94.92, 29.38),   # Texas City
    "5310": (-94.79, 29.30),   # Galveston
    "5311": (-95.32, 28.94),   # Freeport
    "5312": (-97.39, 27.81),   # Corpus Christi
    "5313": (-96.62, 28.62),   # Port Lavaca
}

# This notebook has Atlantic HURDAT2 only. These ports need East/Central Pacific HURDAT2.
OUTSIDE_ATLANTIC_HURDAT = {
    "2501", "2704", "2709", "2713", "2811", "2812",
    "2904", "2905", "2908", "3001", "3002", "3201",
}

assert set(PORT_REFERENCE_POINTS) == set(MAPPED_PORTS), (
    "Port-coordinate coverage differs from the 53-port Census universe"
)

port_reference_df = pl.DataFrame(
    [
        {
            "PORT": port,
            "port_longitude": lon,
            "port_latitude": lat,
            "hurdat_atlantic_covered": port not in OUTSIDE_ATLANTIC_HURDAT,
        }
        for port, (lon, lat) in PORT_REFERENCE_POINTS.items()
    ]
).sort("PORT")

port_reference_path = INTERIM_DIR / "port_reference_points_53.parquet"
port_reference_df.write_parquet(port_reference_path)
print(f"Port reference points saved -> {port_reference_path}")
print(port_reference_df.group_by("hurdat_atlantic_covered").len().sort("hurdat_atlantic_covered"))


In [ ]:
# Build auditable port x storm x month and port x month HURDAT2 exposure tables.
assert HURDAT_PATH.exists(), f"Missing HURDAT2 file: {HURDAT_PATH}"
hurdat = (
    pl.read_parquet(HURDAT_PATH)
    .filter(pl.col("season").is_between(min(STUDY_YEARS), max(STUDY_YEARS)))
    .sort(["storm_id", "datetime_utc"])
)

required_hurdat_cols = {
    "storm_id", "storm_name", "datetime_utc", "latitude", "longitude",
    "max_wind_kt", "r34_ne_nm", "r34_se_nm", "r34_sw_nm", "r34_nw_nm",
}
missing_hurdat_cols = required_hurdat_cols - set(hurdat.columns)
assert not missing_hurdat_cols, f"Missing HURDAT2 columns: {sorted(missing_hurdat_cols)}"

TRACK_FALLBACK_NM = 100.0
EARTH_RADIUS_NM = 3440.065

storm_lat = np.radians(hurdat["latitude"].to_numpy())
storm_lon = np.radians(hurdat["longitude"].to_numpy())
max_wind = hurdat["max_wind_kt"].to_numpy()
r34_ne = hurdat["r34_ne_nm"].to_numpy()
r34_se = hurdat["r34_se_nm"].to_numpy()
r34_sw = hurdat["r34_sw_nm"].to_numpy()
r34_nw = hurdat["r34_nw_nm"].to_numpy()
all_r34_missing = np.isnan(r34_ne) & np.isnan(r34_se) & np.isnan(r34_sw) & np.isnan(r34_nw)

port_track_parts: list[pl.DataFrame] = []
for row in port_reference_df.filter("hurdat_atlantic_covered").iter_rows(named=True):
    port_lat = np.radians(row["port_latitude"])
    port_lon = np.radians(row["port_longitude"])

    # Great-circle distance from each storm center to this port.
    delta_lat = port_lat - storm_lat
    delta_lon = port_lon - storm_lon
    hav = (
        np.sin(delta_lat / 2.0) ** 2
        + np.cos(storm_lat) * np.cos(port_lat) * np.sin(delta_lon / 2.0) ** 2
    )
    hav = np.clip(hav, 0.0, 1.0)
    distance_nm = 2.0 * EARTH_RADIUS_NM * np.arctan2(np.sqrt(hav), np.sqrt(1.0 - hav))

    # Initial bearing FROM storm center TO port selects the relevant wind quadrant.
    bearing_y = np.sin(delta_lon) * np.cos(port_lat)
    bearing_x = (
        np.cos(storm_lat) * np.sin(port_lat)
        - np.sin(storm_lat) * np.cos(port_lat) * np.cos(delta_lon)
    )
    bearing_deg = (np.degrees(np.arctan2(bearing_y, bearing_x)) + 360.0) % 360.0
    selected_r34_nm = np.select(
        [bearing_deg < 90.0, bearing_deg < 180.0, bearing_deg < 270.0],
        [r34_ne, r34_se, r34_sw],
        default=r34_nw,
    )

    r34_exposure = (
        np.isfinite(selected_r34_nm)
        & (selected_r34_nm > 0.0)
        & (distance_nm <= selected_r34_nm)
    )
    near_track_100nm = (max_wind >= 34) & (distance_nm <= TRACK_FALLBACK_NM)
    track_fallback = all_r34_missing & near_track_100nm
    primary_exposure = r34_exposure | track_fallback
    keep = primary_exposure | near_track_100nm

    if keep.any():
        port_track_parts.append(
            hurdat.with_columns(
                pl.lit(row["PORT"]).alias("PORT"),
                pl.Series("port_distance_nm", distance_nm),
                pl.Series("bearing_storm_to_port_deg", bearing_deg),
                pl.Series("selected_r34_nm", selected_r34_nm),
                pl.Series("hurdat_r34_exposure", r34_exposure),
                pl.Series("hurdat_track_fallback", track_fallback),
                pl.Series("hurdat_near_track_100nm", near_track_100nm),
                pl.Series("hurdat_storm_exposure", primary_exposure),
            )
            .filter(pl.Series(keep))
            .select(
                "PORT", "storm_id", "storm_name", "datetime_utc", "status",
                "latitude", "longitude", "max_wind_kt", "port_distance_nm",
                "bearing_storm_to_port_deg", "selected_r34_nm",
                "hurdat_r34_exposure", "hurdat_track_fallback",
                "hurdat_near_track_100nm", "hurdat_storm_exposure",
            )
        )

port_track_points = pl.concat(port_track_parts, how="vertical")
primary_track_points = (
    port_track_points
    .filter("hurdat_storm_exposure")
    .with_columns(pl.col("datetime_utc").dt.strftime("%Y-%m").alias("time"))
)

port_storm_exposure = (
    primary_track_points
    .group_by(["PORT", "time", "storm_id", "storm_name"])
    .agg(
        pl.col("datetime_utc").min().alias("first_exposure_utc"),
        pl.col("datetime_utc").max().alias("last_exposure_utc"),
        pl.col("port_distance_nm").min().alias("min_track_distance_nm"),
        pl.col("max_wind_kt").max().alias("max_wind_kt_at_exposure"),
        pl.len().alias("n_exposed_track_points"),
        pl.col("hurdat_r34_exposure").any(),
        pl.col("hurdat_track_fallback").any(),
    )
    .with_columns(
        pl.when("hurdat_r34_exposure")
        .then(pl.lit("r34_wind_field"))
        .otherwise(pl.lit("track_100nm_fallback"))
        .alias("hurdat_exposure_method")
    )
    .sort(["time", "PORT", "storm_id"])
)

hurdat_primary_month = (
    port_storm_exposure
    .group_by(["PORT", "time"])
    .agg(
        pl.col("storm_id").n_unique().cast(pl.Int16).alias("n_hurdat_storms"),
        pl.col("storm_id").unique().sort().str.join("|").alias("hurdat_storm_ids"),
        pl.col("storm_name").unique().sort().str.join("|").alias("hurdat_storm_names"),
        pl.col("first_exposure_utc").min().alias("hurdat_first_exposure_utc"),
        pl.col("last_exposure_utc").max().alias("hurdat_last_exposure_utc"),
        pl.col("min_track_distance_nm").min().alias("hurdat_min_track_distance_nm"),
        pl.col("max_wind_kt_at_exposure").max().alias("hurdat_max_wind_kt"),
        pl.col("n_exposed_track_points").sum().alias("hurdat_n_exposed_track_points"),
        pl.col("hurdat_r34_exposure").any().cast(pl.Int8).alias("hurdat_r34_month"),
        pl.col("hurdat_track_fallback").any().cast(pl.Int8).alias("hurdat_track_fallback_month"),
    )
    .with_columns(pl.lit(1).cast(pl.Int8).alias("hurdat_storm_month"))
)

hurdat_near_track_month = (
    port_track_points
    .filter("hurdat_near_track_100nm")
    .with_columns(pl.col("datetime_utc").dt.strftime("%Y-%m").alias("time"))
    .group_by(["PORT", "time"])
    .agg(
        pl.col("storm_id").n_unique().cast(pl.Int16).alias("n_hurdat_near_track_storms"),
        pl.col("port_distance_nm").min().alias("hurdat_near_track_min_distance_nm"),
    )
    .with_columns(pl.lit(1).cast(pl.Int8).alias("hurdat_near_track_100nm_month"))
)

hurdat_by_port = (
    hurdat_primary_month
    .join(hurdat_near_track_month, on=["PORT", "time"], how="full", coalesce=True)
    .sort(["PORT", "time"])
)

event_exposure_path = INTERIM_DIR / "hurdat_port_storm_exposure_2013_2025.parquet"
month_exposure_path = INTERIM_DIR / "hurdat_port_month_exposure_2013_2025.parquet"
port_storm_exposure.write_parquet(event_exposure_path)
hurdat_by_port.write_parquet(month_exposure_path)

print(f"HURDAT2 track observations : {hurdat.height:,}")
print(f"Exposed port x storm x month rows : {port_storm_exposure.height:,}")
print(f"Primary exposed port x month rows : {hurdat_primary_month.height:,}")
print(f"Event audit table saved -> {event_exposure_path}")
print(f"Monthly exposure table saved -> {month_exposure_path}")
port_storm_exposure.filter(pl.col("time") == "2015-06").sort(["storm_id", "PORT"])


In [ ]:
# Left-join broad NOAA indicators and port-specific HURDAT2 exposure.
panel = (
    imports_raw
    .join(storms_by_port, on=["time", "PORT"], how="left")
    .join(port_reference_df, on="PORT", how="left")
    .join(hurdat_by_port, on=["time", "PORT"], how="left")
    .with_columns(
        pl.col("storm_month").fill_null(0).cast(pl.Int8),
        pl.col("n_storm_events").fill_null(0).cast(pl.Int32),
        pl.col("total_property_damage_usd").fill_null(0.0),
        pl.col("total_crop_damage_usd").fill_null(0.0),
        pl.col("hurdat_storm_month").fill_null(0).cast(pl.Int8),
        pl.col("hurdat_r34_month").fill_null(0).cast(pl.Int8),
        pl.col("hurdat_track_fallback_month").fill_null(0).cast(pl.Int8),
        pl.col("hurdat_near_track_100nm_month").fill_null(0).cast(pl.Int8),
        pl.col("n_hurdat_storms").fill_null(0).cast(pl.Int16),
        pl.col("n_hurdat_near_track_storms").fill_null(0).cast(pl.Int16),
        pl.col("hurdat_n_exposed_track_points").fill_null(0).cast(pl.Int32),
    )
    .sort(["time", "PORT", "I_COMMODITY"])
)

print(f"Panel shape                  : {panel.shape}")
print(f"Time range                   : {panel['time'].min()} → {panel['time'].max()}")
print(f"Unique PORT × HS4 × month    : {panel.n_unique(subset=['time', 'PORT', 'I_COMMODITY']):,}")
print(f"NOAA state storm observations: {panel['storm_month'].sum():,}")
print(f"HURDAT port exposure obs.    : {panel['hurdat_storm_month'].sum():,}")
print(f"  (% of all observations)    : {panel['hurdat_storm_month'].mean() * 100:.2f}%")
panel.head(3)


In [ ]:
out_path = INTERIM_DIR / "panel_port_hs4_month_2013_2025.parquet"
panel.write_parquet(out_path)
print(f"Panel saved → {out_path}")
print(f"File size    : {out_path.stat().st_size / 1e6:.1f} MB")


## 4. Missing Value Analysis

In [ ]:
# ── Panel null counts ─────────────────────────────────────────────────────────
print("=== Panel null counts ===")
print(panel.null_count())

# Import value completeness
n_null_val = panel.filter(pl.col("GEN_VAL_MO").is_null()).height
print(f"\nRows with null GEN_VAL_MO    : {n_null_val:,}  ({n_null_val / panel.height * 100:.2f}%)")

# Months per port: check for gaps in the time series
print("\n=== Expected vs. observed months per port ===")
expected_months = len(STUDY_YEARS) * 12
port_month_counts = (
    panel
    .select(["PORT", "PORT_NAME", "time"])
    .unique()
    .group_by(["PORT", "PORT_NAME"])
    .agg(pl.len().alias("observed_months"))
    .with_columns((pl.col("observed_months") / expected_months * 100).round(1).alias("coverage_pct"))
    .sort("PORT")
)
print(port_month_counts)

# Storm-month coverage by port
print("\n=== Storm-month coverage by port ===")
port_storm = (
    panel
    .select(["PORT", "PORT_NAME", "time", "storm_month"])
    .unique()
    .group_by(["PORT", "PORT_NAME"])
    .agg(
        pl.len().alias("total_port_months"),
        pl.col("storm_month").max().alias("has_storm_data"),
        pl.col("storm_month").sum().alias("storm_months"),
    )
    .with_columns(
        (pl.col("storm_months") / pl.col("total_port_months") * 100).round(2).alias("storm_pct")
    )
    .sort("storm_pct", descending=True)
)
print(port_storm)


## 5. Event Study: Hurricane Harvey (August 2017)

Hurricane Harvey made landfall near Rockport, TX on 25 August 2017 — one of the costliest Atlantic hurricanes on record. Houston (port 5301) and Galveston (5310) are the two major Census ports in Texas. We plot total monthly import values for an 11-month window (January–November 2017) to visualise the before/after pattern.

In [ ]:
HARVEY_MONTH  = "2017-08"
HARVEY_PORTS  = {"5301": "Houston, TX", "5310": "Galveston, TX"}

# 11-month window: Jan 2017 – Nov 2017
WINDOW = [f"2017-{m:02d}" for m in range(1, 12)]

# Confirm Harvey appears in the NOAA data
print("=== NOAA events in Texas, August 2017 ===")
texas_harvey = (
    storms_raw
    .filter((pl.col("state") == "TEXAS") & (pl.col("time") == HARVEY_MONTH))
    .select(["event_type", "cz_name", "property_damage_usd"])
    .sort("property_damage_usd", descending=True, nulls_last=True)
)
print(texas_harvey)

# Aggregate across all HS4 codes for each target port × month
harvey_data = (
    panel
    .filter(pl.col("PORT").is_in(list(HARVEY_PORTS.keys())) & pl.col("time").is_in(WINDOW))
    .group_by(["time", "PORT", "PORT_NAME"])
    .agg(
        pl.col("GEN_VAL_MO").sum().alias("total_import_value"),
        pl.col("n_storm_events").max().alias("n_storm_events"),
        pl.col("hurdat_storm_month").max().alias("is_hurdat_exposed"),
    )
    .sort(["PORT", "time"])
)

print(f"\nHarvey window shape: {harvey_data.shape}")
print(harvey_data.select(["time", "PORT", "PORT_NAME", "total_import_value", "is_hurdat_exposed"]))


In [ ]:
fig, axes = plt.subplots(len(HARVEY_PORTS), 1, figsize=(11, 8), sharex=True)
fig.suptitle(
    "Monthly Import Value Before & After Hurricane Harvey (Aug 2017)\n"
    "Source: U.S. Census International Trade API  ·  Atlantic HURDAT2",
    fontsize=12, fontweight="bold",
)

for ax, (port_code, port_label) in zip(axes, HARVEY_PORTS.items()):
    port_df = harvey_data.filter(pl.col("PORT") == port_code).sort("time")
    months = port_df["time"].to_list()
    values = [v / 1e9 if v is not None else 0.0 for v in port_df["total_import_value"].to_list()]
    flags  = port_df["is_hurdat_exposed"].to_list()

    colors = ["#c0392b" if f else "#2471a3" for f in flags]
    ax.bar(months, values, color=colors, alpha=0.82, width=0.6, zorder=3)

    # Dashed reference line at the Harvey month
    ax.axvline(x=HARVEY_MONTH, color="#922b21", linewidth=1.4, linestyle="--", zorder=4, alpha=0.65)

    # Annotate specifically the Harvey landfall month (2017-08)
    if HARVEY_MONTH in months:
        harvey_idx = months.index(HARVEY_MONTH)
        ax.annotate(
            "Harvey\nlandfall\n(Aug 2017)",
            xy=(months[harvey_idx], values[harvey_idx]),
            xytext=(0, 14), textcoords="offset points",
            ha="center", fontsize=8.5, color="#922b21", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#922b21", lw=1.1),
        )

    ax.set_title(f"{port_label}  (port code {port_code})", fontsize=10.5, loc="left")
    ax.set_ylabel("Import Value (USD bn)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}B"))
    ax.tick_params(axis="x", rotation=40, labelsize=9)
    ax.grid(axis="y", alpha=0.35, zorder=0)
    ax.legend(
        handles=[
            Patch(facecolor="#c0392b", alpha=0.82, label="Port inside HURDAT2 exposure field"),
            Patch(facecolor="#2471a3", alpha=0.82, label="No port-specific HURDAT2 exposure"),
        ],
        fontsize=9, loc="upper left",
    )

fig.tight_layout(rect=[0, 0, 1, 0.96])
# out_fig = FIGURES_DIR / "harvey_2017_port_imports.png"
# plt.savefig(out_fig, bbox_inches="tight", dpi=150)
plt.show()
# print(f"Figure saved → {out_fig}")
